In [ ]:
from deap import base, creator, tools, algorithms
import random
import numpy

In [ ]:
import random
from deap import base, creator, tools

IND_SIZE = 100
POP_SIZE = 300
CX_PB = 0.5
MUT_PB = 0.2
NGEN = 40
TOURNSIZE = 3

creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()
toolbox.register("attr_bool", random.randint, 0, 1)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_bool, n=IND_SIZE)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

def evalOneMax(individual):
    return sum(individual),

toolbox.register("evaluate", evalOneMax)
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutFlipBit, indpb=1.0/IND_SIZE)
toolbox.register("select", tools.selTournament, tournsize=TOURNSIZE)

In [ ]:
from deap import base, creator

creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

In [ ]:
def evalOneMax(individual):
    return sum(individual),

In [ ]:
toolbox.register("mate", tools.cxTwoPoint)

In [ ]:
toolbox.register("mutate", tools.mutFlipBit, indpb=0.05)

In [ ]:
toolbox.register("select", tools.selTournament, tournsize=TOURNSIZE)

In [ ]:
import random
from deap import base, creator, tools

creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()
toolbox.register("attr_bool", random.randint, 0, 1)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_bool, n=100)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

In [ ]:
import random
from deap import base, creator, tools

creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()
toolbox.register("attr_bool", random.randint, 0, 1)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_bool, n=100)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

def evalOneMax(individual):
    return sum(individual),

toolbox.register("evaluate", evalOneMax)
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutFlipBit, indpb=0.05)
toolbox.register("select", tools.selTournament, tournsize=3)

In [ ]:
import random
import numpy as np
from deap import base, creator, tools, algorithms

def main_algorithm(pop_size=300, n_gen=40, cx_pb=0.7, mut_pb=0.2, ind_pb=0.05, n_bits=100, seed=42, parallel=False, n_workers=4):
    random.seed(seed)
    np.random.seed(seed)

    if parallel:
        from multiprocessing import Pool
        pool = Pool(processes=n_workers)
        toolbox = base.Toolbox()
        toolbox.register("map", pool.map)
    else:
        toolbox = base.Toolbox()

    creator.create("FitnessMax", base.Fitness, weights=(1.0,))
    creator.create("Individual", list, fitness=creator.FitnessMax)

    toolbox.register("attr_bool", random.randint, 0, 1)
    toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_bool, n=n_bits)
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)
    toolbox.register("evaluate", lambda ind: (sum(ind),))
    toolbox.register("mate", tools.cxTwoPoint)
    toolbox.register("mutate", tools.mutFlipBit, indpb=ind_pb)
    toolbox.register("select", tools.selTournament, tournsize=3)

    pop = toolbox.population(n=pop_size)
    hof = tools.HallOfFame(1)
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("avg", np.mean)
    stats.register("std", np.std)
    stats.register("min", np.min)
    stats.register("max", np.max)

    logbook = tools.Logbook()
    logbook.header = ['gen', 'nevals'] + stats.fields

    invalid_ind = [ind for ind in pop if not ind.fitness.valid]
    fitnesses = toolbox.map(toolbox.evaluate, invalid_ind)
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit

    hof.update(pop)
    record = stats.compile(pop)
    logbook.record(gen=0, nevals=len(invalid_ind), **record)

    for gen in range(1, n_gen + 1):
        offspring = algorithms.varAnd(pop, toolbox, cx_pb, mut_pb)
        
        invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
        fitnesses = toolbox.map(toolbox.evaluate, invalid_ind)
        for ind, fit in zip(invalid_ind, fitnesses):
            ind.fitness.values = fit

        pop = toolbox.select(offspring, k=len(pop))
        hof.update(pop)
        
        record = stats.compile(pop)
        logbook.record(gen=gen, nevals=len(invalid_ind), **record)

    if parallel:
        pool.close()
        pool.join()

    return pop, logbook, hof

if __name__ == "__main__":
    pop, log, hof = main_algorithm(parallel=False)
    print(f"Best fitness: {hof[0].fitness.values[0]}")
    print(log)

In [ ]:
import numpy
from deap import tools

stats = tools.Statistics(key=lambda ind: ind.fitness.values)
stats.register("avg", numpy.mean)
stats.register("std", numpy.std)
stats.register("min", numpy.min)
stats.register("max", numpy.max)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(gen, avg, label='Average Fitness', color='blue')
plt.plot(gen, max, label='Max Fitness', color='red')
plt.xlabel('Generation')
plt.ylabel('Fitness')
plt.title('OneMax Problem - Fitness over Generations')
plt.legend()
plt.grid(True)
plt.show()